## Librerías

In [20]:
from lime import submodular_pick
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
import random
from src.fuzzy_LIME import *

import warnings
warnings.filterwarnings('ignore')

## Prueba de generación de variables

In [8]:
def generar_dataset_polinomico(n_instancias=1000, semilla=None):
    """
    Genera un dataset sintético con un número dinámico de variables y un target 
    basado en un polinomio generado aleatoriamente.
    """
    if semilla is not None:
        np.random.seed(semilla)
        
    # 1. Generar los nombres de las columnas (x1, x2, x3, ...)
    columnas = [f'x{i}' for i in range(1, 7)]
    
    # 2. Generar la matriz de datos de entrada (features)
    # Rango [1, 3] para que las potencias altas no disparen los valores al infinito
    X = np.random.uniform(1, 3, size=(n_instancias, 6))
    
    # 3. Generar aleatoriamente los coeficientes y las potencias para el polinomio
    # Coeficientes entre -3 y 3
    coeficientes = [-1, 2, -3, 3, 1, -2] # np.random.randint(-3, 4, size=6)
    # Potencias elegidas manualmente para conocer el polinomio exacto 
    potencias = [2,1,5,5,2,1] # np.random.randint(1, 5, size=n_variables)
    
    # 4. Calcular la variable de salida (target) 'y' y construir la fórmula en texto
    y = np.zeros(n_instancias)
    terminos_formula = []
    
    for i in range(6):
        # Fórmula: c * (x^p)
        y += coeficientes[i] * (X[:, i] ** potencias[i])
        
        # Guardamos el término en texto para saber cuál es el "ground truth"
        termino_str = f"{coeficientes[i]}*{columnas[i]}^{potencias[i]}"
        terminos_formula.append(termino_str)
        
    # Limpiamos un poco la fórmula generada para que se lea mejor matemáticamente
    formula_final = "y = " + " + ".join(terminos_formula).replace("+ -", "- ")
    
    # 5. Crear el DataFrame
    df = pd.DataFrame(X, columns=columnas)
    df['target'] = y
    
    return df, formula_final, coeficientes, potencias

In [11]:
# Vamos a generar un dataset con 6 variables
SEED = 1234
dataset, formula_real, coeficientes, potencias = generar_dataset_polinomico(n_instancias=5000, semilla=SEED) # n_variables=6, 

np.random.seed(SEED)
random.seed(SEED)

# Imprimir la fórmula que el código se ha inventado para este dataset
print("La función real (Ground Truth) subyacente es:")
print(formula_real)
print("-" * 60)

# Mostrar el dataset resultante
display(dataset.head(20))

La función real (Ground Truth) subyacente es:
y = -1*x1^2 + 2*x2^1 - 3*x3^5 + 3*x4^5 + 1*x5^2 - 2*x6^1
------------------------------------------------------------


,x1,x2,x3,x4,x5,x6,target
0,1.383039,2.244218,1.875455,2.570717,2.559952,1.545185,273.247302
1,1.552929,2.603744,2.916279,2.751865,1.715635,2.001990,-157.633124
2,2.366926,2.425404,1.740502,2.122392,2.006166,1.027537,82.496865
3,2.545653,2.765282,1.729772,2.230792,1.150762,1.737648,116.176689
4,2.866280,2.302756,1.794405,2.577460,1.633672,2.136197,280.231612
5,2.738255,1.872347,2.604295,1.287534,2.408522,2.409163,-351.551121
6,1.437584,2.849735,1.884282,2.818632,1.119618,1.368574,464.608017
7,1.094711,2.349762,2.189250,2.066620,1.086648,2.122866,-37.341822
8,1.659337,2.005934,1.223789,2.214387,2.131889,1.013528,155.272583
9,2.234883,2.824246,2.581048,2.984163,2.917604,2.583928,370.319837


## Division train test

In [12]:
# 1. Preparar los datos
X = dataset.drop('target', axis=1)
y = dataset['target']

# Dividir en entrenamiento y test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

## Random Forest

In [13]:
#grid search para encontrar el mejor modelo de random forest 
grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

In [14]:
search_rf = GridSearchCV(
    estimator=RandomForestRegressor(random_state=SEED), 
    param_grid=grid_rf, 
    cv=5, 
    n_jobs=-1, 
    verbose=2
)
search_rf.fit(X_train, y_train)

Fitting 5 folds for each of 81 candidates, totalling 405 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestR...om_state=1234)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [None, 10, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], 'n_estimators': [50, 100, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and par

In [15]:
rf = search_rf.best_estimator_
y_pred = rf.predict(X_test)

In [16]:
# 1. El usuario importa tu herramienta
# from fuzzy_lime import FuzzyLimeExplainer

# 2. INICIALIZAR Y ENTRENAR (El usuario no ve las matemáticas, solo invoca la herramienta)
# num_instancias_a_procesar: puedes omitirlo o poner len(X_train) para entrenamiento completo.
explicador = FuzzyLimeExplainer(
    training_data=X_train, 
    predict_fn=rf.predict,
    max_depth=3,            # Soporta cualquier profundidad gracias a la fase 3
    num_clusters=8          # Límite máximo de reglas globales legibles
)

# Ejecuta todo el pipeline (Fases 1 a 5) y se guarda el conocimiento internamente
explicador.fit()  

Iniciando entrenamiento de Fuzzy-LIME (4000 instancias)...
✅ Entrenamiento completado. Modelo Subrogado encapsulado en 8 reglas maestras.


In [17]:
# 3. EXPLICAR NUEVAS INSTANCIAS (Se ejecuta al instante)
fila_nueva = X_test.iloc[0]
explicacion = explicador.explain_instance(fila_nueva)

# 4. MOSTRAR RESULTADO
explicacion.show_in_notebook()

### Explicación global con SP-LIME

In [22]:
# Configuración
num_explicaciones_representativas = 5  # Cantidad de instancias a mostrar para entender el global
num_instancias_a_muestrear = len(X_test) #100 # Muestra aleatoria de X_test a evaluar (mayor = mejor, pero más lento)

# 1. Ejecutar el algoritmo Submodular Pick
sp_explainer = submodular_pick.SubmodularPick(
    explainer=explicador,
    data=X_test.values,         # Es recomendable pasar los valores como array de numpy
    predict_fn=rf.predict,      # Pasamos el modelo Random Forest
    sample_size=num_instancias_a_muestrear,
    num_features=len(X_test.columns),
    num_exps_desired=num_explicaciones_representativas
)

# 2. Visualizar las explicaciones globales (el conjunto representativo)
for i, exp in enumerate(sp_explainer.sp_explanations):
    print(f"\n{'='*60}")
    print(f"Explicación Global/Representativa {i+1}")
    print(f"{'='*60}")
    exp.show_in_notebook(show_table=True)

TypeError: FuzzyLimeExplainer.explain_instance() got an unexpected keyword argument 'num_features'